# Week 3 — Train models, compare them, visualize

Hi team — we fit **Ridge** (simple baseline) and **Random Forest** (nonlinear) on the same Week 2 features.

**Time test:** older months train, newer months test → forecasting story.

**Spatial test:** hide whole stations → “new site” story.

**Outputs:** same CSV paths as before (`week3_model_metrics.csv`, `week3_predictions.csv`) so Week 4 still runs.

**Plots + winner:** we print **average RMSE** (time + spatial) and bar charts / scatters so the deck isn’t only tables.


### Imports + pipelines

**Pipeline** = impute → scale / one-hot → model. Keeps preprocessing consistent when we refactor to cross-validation later.

Plot style: try seaborn grid, fall back if your matplotlib is minimal.


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    try:
        plt.style.use("seaborn-whitegrid")
    except OSError:
        pass
print("Libraries loaded")


### Load `week2_features.csv`

If missing, run Week 2 first.


In [ ]:
ROOT = Path(".")
INPUT = ROOT / "datasets" / "week2_features.csv"
PRED_OUT = ROOT / "datasets" / "week3_predictions.csv"
METRIC_OUT = ROOT / "datasets" / "week3_model_metrics.csv"

df = pd.read_csv(INPUT, parse_dates=["date"])
print("Loaded:", INPUT, "| shape:", df.shape)
display(df.head())


### Station id for grouping only

`openaq_id` is the EEA station code. We use it for spatial split but **do not** put it in `X` as a feature (that would be memorizing ids).


In [ ]:
if "openaq_id" in df.columns:
    df["station_group"] = df["openaq_id"].astype(str)
else:
    df["station_group"] = "unknown"
print("Stations:", df["station_group"].nunique())


### Build `X` and `y`

Target = this month’s `PM2_5`. Features = everything else except `date` and `openaq_id`.


In [ ]:
target_col = "PM2_5"
ignore = {"PM2_5", "date", "openaq_id"}
model_features = [c for c in df.columns if c not in ignore]
X = df[model_features].copy()
y = df[target_col].copy()
groups = df["station_group"]
print("n_features:", len(model_features))


### Time split (80% of timeline train, 20% test)

Same spirit as Week 2. Good for “how well do we predict upcoming months?”


In [ ]:
cutoff_date = df["date"].quantile(0.8)
time_train_mask = df["date"] <= cutoff_date
time_test_mask = df["date"] > cutoff_date
X_train_t, X_test_t = X.loc[time_train_mask], X.loc[time_test_mask]
y_train_t, y_test_t = y.loc[time_train_mask], y.loc[time_test_mask]
print("Cutoff:", cutoff_date.date(), "| train/test:", len(X_train_t), len(X_test_t))


### Spatial split (20% of stations held out)

Every row of a station stays on one side. Harder test.


In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(splitter.split(X, y, groups=groups))
X_train_s, X_test_s = X.iloc[tr_idx], X.iloc[te_idx]
y_train_s, y_test_s = y.iloc[tr_idx], y.iloc[te_idx]
print("Spatial train/test rows:", len(X_train_s), len(X_test_s))


### Preprocessing

Numeric: median impute + scale. Categorical (if any): most-frequent + one-hot. Shared by Ridge and RF for a fair comparison.


In [ ]:
cat_cols = [c for c in model_features if X[c].dtype == "object"]
num_cols = [c for c in model_features if c not in cat_cols]
pre = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])
print("Numeric:", len(num_cols), "| Categorical:", len(cat_cols))


### Fit Ridge (time train)


In [ ]:
ridge = Pipeline([("preprocess", pre), ("model", Ridge(alpha=1.0, random_state=42))])
ridge.fit(X_train_t, y_train_t)
print("Ridge OK")


### Fit Random Forest (time train)


In [ ]:
rf = Pipeline([
    ("preprocess", pre),
    ("model", RandomForestRegressor(
        n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1,
    )),
])
rf.fit(X_train_t, y_train_t)
print("Random Forest OK")


### Metrics on **time** test


In [ ]:
def reg_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "R2": r2_score(y_true, y_pred),
    }

pred_ridge_t = ridge.predict(X_test_t)
pred_rf_t = rf.predict(X_test_t)
m_ridge_t = reg_metrics(y_test_t, pred_ridge_t)
m_rf_t = reg_metrics(y_test_t, pred_rf_t)
print("Ridge  time:", m_ridge_t)
print("RF     time:", m_rf_t)


### Refit both on spatial train → evaluate spatial test


In [ ]:
ridge_s = Pipeline([("preprocess", pre), ("model", Ridge(alpha=1.0, random_state=42))])
rf_s = Pipeline([
    ("preprocess", pre),
    ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
])
ridge_s.fit(X_train_s, y_train_s)
rf_s.fit(X_train_s, y_train_s)
pred_ridge_s = ridge_s.predict(X_test_s)
pred_rf_s = rf_s.predict(X_test_s)
m_ridge_s = reg_metrics(y_test_s, pred_ridge_s)
m_rf_s = reg_metrics(y_test_s, pred_rf_s)
print("Ridge  spatial:", m_ridge_s)
print("RF     spatial:", m_rf_s)


### Save metrics table


In [ ]:
metrics = pd.DataFrame([
    {"split": "time", "model": "Ridge", **m_ridge_t},
    {"split": "time", "model": "RandomForest", **m_rf_t},
    {"split": "spatial", "model": "Ridge", **m_ridge_s},
    {"split": "spatial", "model": "RandomForest", **m_rf_s},
])
display(metrics)
metrics.to_csv(METRIC_OUT, index=False)
print("Saved:", METRIC_OUT)


### Which model wins? (group rule)

**Primary:** lowest **average RMSE** over *time* and *spatial* rows in `metrics`.

**Tie-break:** lower average MAE, then higher average R².

Document this sentence in the report so it doesn’t look like we picked favorites by eye.


In [ ]:
def winner_from_metrics(metrics_df):
    t = metrics_df.pivot_table(index="model", columns="split", values="RMSE", aggfunc="first")
    t["avg_RMSE"] = t.mean(axis=1)
    t["avg_MAE"] = metrics_df.pivot_table(index="model", columns="split", values="MAE", aggfunc="first").mean(axis=1)
    t["avg_R2"] = metrics_df.pivot_table(index="model", columns="split", values="R2", aggfunc="first").mean(axis=1)
    t = t.sort_values(["avg_RMSE", "avg_MAE"], ascending=[True, True])
    best = t.index[0]
    return best, t

champion_model, summary = winner_from_metrics(metrics)
print("\n=== Averaged over time + spatial tests ===")
display(summary.round(4))
print("\n*** Champion (lowest avg RMSE):", champion_model, "***")


### Plot — RMSE & MAE bars (lower is better)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, metric in zip(axes, ["RMSE", "MAE"]):
    metrics.pivot(index="model", columns="split", values=metric).plot(kind="bar", ax=ax, rot=0)
    ax.set_title(metric)
    ax.legend(title="split")
plt.tight_layout()
plt.show()


### Plot — actual vs predicted (time test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
mx = float(max(y_test_t.max(), pred_ridge_t.max(), pred_rf_t.max()))
for ax, title, pred in zip(axes, ["Ridge", "Random Forest"], [pred_ridge_t, pred_rf_t]):
    ax.scatter(y_test_t, pred, alpha=0.35, s=12)
    ax.plot([0, mx], [0, mx], "k--", lw=1)
    ax.set_xlabel("Actual PM2_5")
    ax.set_ylabel("Predicted PM2_5")
    ax.set_title(title + " (time test)")
plt.tight_layout()
plt.show()


### Plot — residuals (time test)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True)
for ax, name, pred in zip(axes, ["Ridge", "RF"], [pred_ridge_t, pred_rf_t]):
    r = y_test_t.values - pred
    ax.hist(r, bins=30, color="steelblue", edgecolor="white")
    ax.axvline(0, color="black", linestyle="--")
    ax.set_title(name + " residuals")
    ax.set_xlabel("actual - pred")
plt.tight_layout()
plt.show()


### Plot — Random Forest feature importances (top 15)


In [ ]:
names = rf.named_steps["preprocess"].get_feature_names_out()
imp = pd.Series(rf.named_steps["model"].feature_importances_, index=names).sort_values(ascending=False).head(15)
plt.figure(figsize=(8, 5))
imp.sort_values().plot(kind="barh", color="darkgreen")
plt.title("RF top 15 importances (time-train fit)")
plt.tight_layout()
plt.show()


### Save time-test predictions (both models)


In [ ]:
pred_df = df.loc[time_test_mask, ["date", "PM2_5"]].copy()
if "openaq_id" in df.columns:
    pred_df["openaq_id"] = df.loc[time_test_mask, "openaq_id"].values
pred_df["pred_ridge"] = pred_ridge_t
pred_df["pred_rf"] = pred_rf_t
pred_df.to_csv(PRED_OUT, index=False)
print("Saved:", PRED_OUT)
display(pred_df.head())


### SHAP (optional; needs `shap`)

Dense matrix only — avoids the `isnan` dtype bug on sparse inputs.


In [ ]:
try:
    import scipy.sparse as sp
    import shap
    rf_full = Pipeline([
        ("preprocess", pre),
        ("model", RandomForestRegressor(n_estimators=400, min_samples_leaf=2, random_state=42, n_jobs=-1)),
    ])
    rf_full.fit(X, y)
    Xp = rf_full.named_steps["preprocess"].transform(X)
    if sp.issparse(Xp):
        Xp = Xp.toarray()
    Xp = np.asarray(Xp, dtype=np.float64)
    rng = np.random.RandomState(42)
    n = min(800, Xp.shape[0])
    idx = rng.choice(Xp.shape[0], size=n, replace=False)
    Xs = Xp[idx]
    explainer = shap.TreeExplainer(rf_full.named_steps["model"])
    sv = explainer.shap_values(Xs)
    fn = rf_full.named_steps["preprocess"].get_feature_names_out()
    shap.summary_plot(sv, Xs, feature_names=fn, show=False)
    plt.title("SHAP (RF sample)")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print("SHAP skipped:", e)


### Handoff checklist

- Metrics + predictions saved.
- Champion printed from **avg RMSE** rule.
- Export plots for slides if needed.

Next: **`week4_policy_translation.ipynb`**.
